# 🌸 Seha Personal AI — Kaggle Launcher
Run cells from top to bottom. Add API keys, owner number, and GitHub token only in the configuration cell below. User profiles/settings are stored on GitHub only, never on Kaggle disk. Do not publish this notebook after entering credentials.

In [ ]:
# 1) CONFIGURATION — enter project values and credentials only in this Kaggle cell
# Sri Lanka examples: 0771234567, +94 77 123 4567, +94 077 123 4567, 94771234567.
OWNER_NUMBER = ""       # Owner number (used to authenticate .owner commands)
WHATSAPP_NUMBER = ""    # Exact WhatsApp account to pair; blank = OWNER_NUMBER
DEFAULT_COUNTRY_CODE = "94"
OWNER_NAME = "Owner"
GEMINI_API_KEY = ""     # Primary AI key
GROQ_API_KEY = ""       # Optional fallback key
GITHUB_OWNER = "Sachintha919"
GITHUB_REPO = "Seha-Personal-AI"
GITHUB_TOKEN = ""       # REQUIRED: fine-grained token with Contents Read/Write for this repo
MAX_MEDIA_MB = 15

assert OWNER_NUMBER.strip(), "Enter OWNER_NUMBER first."
if not WHATSAPP_NUMBER.strip():
    WHATSAPP_NUMBER = OWNER_NUMBER
assert GEMINI_API_KEY or GROQ_API_KEY, "Enter at least one AI API key."
assert GITHUB_OWNER.strip() and GITHUB_REPO.strip() and GITHUB_TOKEN.strip(), "GitHub-only storage requires owner, repo, and token."
print("Configuration loaded. Profile/settings storage is GitHub-only; Kaggle files will not be used for user details.")

In [ ]:
# 2) Install Node.js 22 and FFmpeg
import subprocess, os, sys
subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash -", shell=True, check=True)
subprocess.run("apt-get update -qq && apt-get install -y nodejs ffmpeg git -qq", shell=True, check=True)
print(subprocess.check_output(['node','--version'], text=True).strip())
print(subprocess.check_output(['ffmpeg','-version'], text=True).splitlines()[0])

In [ ]:
# 3) Clone the GitHub repository (or update the existing clone)
from pathlib import Path
import base64, os
PROJECT_DIR = Path("/kaggle/working/Seha-Personal-AI")
clone_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"
git_env = os.environ.copy()
if GITHUB_TOKEN.strip():
    git_env["GIT_CONFIG_COUNT"] = "1"
    git_env["GIT_CONFIG_KEY_0"] = "http.extraheader"
    git_env["GIT_CONFIG_VALUE_0"] = "AUTHORIZATION: basic " + base64.b64encode(("x-access-token:" + GITHUB_TOKEN).encode()).decode()
if (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], env=git_env, check=True)
else:
    subprocess.run(["git", "clone", clone_url, str(PROJECT_DIR)], env=git_env, check=True)
print("Repository ready. Local JSON copies remain; Seha uses GitHub API for profile/settings reads and writes.")

In [ ]:
# 4) Install packages and verify source syntax
subprocess.run(['npm','install'], cwd=str(PROJECT_DIR), check=True)
subprocess.run(['npm','run','check'], cwd=str(PROJECT_DIR), check=True)
print('Dependencies installed and syntax checks passed.')

In [ ]:
# 5) Start Seha — pairing code appears in this cell output
env = os.environ.copy()
env.update({
    "BOT_OWNER_NUMBER": OWNER_NUMBER,
    "WHATSAPP_NUMBER": WHATSAPP_NUMBER,
    "OWNER_NAME": OWNER_NAME,
    "DEFAULT_COUNTRY_CODE": DEFAULT_COUNTRY_CODE,
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "GROQ_API_KEY": GROQ_API_KEY,
    "GITHUB_OWNER": GITHUB_OWNER,
    "GITHUB_REPO": GITHUB_REPO,
    "GITHUB_TOKEN": GITHUB_TOKEN,
    "AUTH_DIR": "/kaggle/working/seha-auth",
    "MAX_MEDIA_MB": str(MAX_MEDIA_MB)
})
subprocess.run(["node", "src/main.js"], cwd=str(PROJECT_DIR), env=env, check=False)

## Pair your WhatsApp account
When the 8-digit code appears, use the **same WhatsApp account whose number you entered in `WHATSAPP_NUMBER`**.
**WhatsApp → Settings → Linked Devices → Link a Device → Link with phone number instead** → enter the code.

Sri Lankan examples accepted: `0771234567`, `+94 77 123 4567`, `+94 077 123 4567`, `94771234567`. The number must already be registered with WhatsApp.

The cloned repository may contain `database/users.json` and `database/settings.json`, but the bot does not use the local copies. It reads and writes these records directly through GitHub API; user details and settings are not written to Kaggle files. If GitHub is unavailable or the token lacks permission, saving fails rather than using local storage.

API keys, owner number, and GitHub token are supplied only in this Kaggle cell. WhatsApp auth files stay in `/kaggle/working/seha-auth` and are not uploaded to GitHub; if Kaggle resets that folder, pair again.